# Predict Customer Churn — RealMLP Standalone

Self-contained notebook using **RealMLP** (`rtdl_revisiting_models`) for the `playground-series-s6e3` competition.

Pipeline:
1. Install dependencies
2. Load & preprocess data
3. Feature engineering
4. Optuna hyperparameter tuning (optional)
5. Multi-seed ensemble (5 seeds × 10 folds = 50 models)
6. Save submission + OOF

In [ ]:
# ── Install dependencies (run once) ──────────────────────────────────────────
import subprocess, sys

def pip_install(*args):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *args])

# P100 = sm_60 — requires cu118 (CUDA 12.x builds dropped sm_60 support)
import torch
if torch.cuda.is_available():
    cap = torch.cuda.get_device_capability()
    if cap[0] < 7:   # sm_60 (P100), sm_61, sm_62, sm_70 = V100 is fine on cu121
        print(f'GPU compute capability {cap[0]}.{cap[1]} detected — installing cu118 build for sm_60 support ...')
        pip_install('torch', 'torchvision', 'torchaudio',
                    '--index-url', 'https://download.pytorch.org/whl/cu118')

pip_install('rtdl_revisiting_models', 'optuna', 'tqdm')
print('Dependencies ready.')

In [ ]:
import os, gc, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from rtdl_revisiting_models import MLP
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
import optuna
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

# ── Environment Detection ────────────────────────────────────────────────────
IS_KAGGLE = os.path.exists('/kaggle/input')
DATA_DIR  = '/kaggle/input/playground-series-s6e3/' if IS_KAGGLE else '../data/'

def _cuda_works():
    """Return True only if CUDA is present AND the kernel image is compatible."""
    if not torch.cuda.is_available():
        return False
    try:
        t = torch.tensor([1.0]).cuda()
        _ = t + t          # triggers a real kernel call
        del t
        return True
    except Exception as e:
        print(f'[WARN] CUDA detected but unusable: {e}')
        print('       Falling back to CPU.')
        print('       To fix, reinstall PyTorch for your CUDA version:')
        print('       https://pytorch.org/get-started/locally/')
        return False

if _cuda_works():
    DEVICE = 'cuda'
elif torch.backends.mps.is_available():
    DEVICE = 'mps'
else:
    DEVICE = 'cpu'

print(f'Environment : {"Kaggle" if IS_KAGGLE else "Local"}')
print(f'Device      : {DEVICE}')
print(f'CUDA avail  : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU         : {torch.cuda.get_device_name(0)}')
    print(f'CUDA ver    : {torch.version.cuda}')
print(f'PyTorch ver : {torch.__version__}')
print(f'Data dir    : {DATA_DIR}')

In [ ]:
# ── CUDA Compatibility Check & Fix ───────────────────────────────────────────
# Run this cell if you hit:
#   "CUDA error: no kernel image is available for execution on the device"
#
# Root cause: PyTorch was compiled for a different GPU architecture.
# Fix: reinstall PyTorch matching your CUDA driver.

import subprocess as _sp

def _get_driver_version():
    try:
        out = _sp.check_output(['nvidia-smi', '--query-gpu=driver_version',
                                '--format=csv,noheader'], text=True)
        return out.strip().split('\n')[0]
    except Exception:
        return None

driver = _get_driver_version()
import torch as _torch
torch_cuda = _torch.version.cuda or 'N/A'

print(f'NVIDIA driver version : {driver or "not found"}')
print(f'PyTorch CUDA build    : {torch_cuda}')
print(f'PyTorch version       : {_torch.__version__}')

if driver:
    try:
        major = int(driver.split('.')[0])
        if major >= 525:
            cmd = 'pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121'
        elif major >= 450:
            cmd = 'pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118'
        else:
            cmd = 'pip install torch torchvision torchaudio'
        print(f'\nRecommended fix:\n  {cmd}')
        print('\nUncomment the line below to auto-reinstall (then restart kernel):')
        print('# _sp.check_call(cmd.split())')
    except Exception:
        pass
else:
    print('\nVisit https://pytorch.org/get-started/locally/ to get the right build.')

In [ ]:
# ── Experiment Settings ───────────────────────────────────────────────────────
RUN_TUNING   = True   # False → skip Optuna, use PRECOMPUTED_PARAMS
N_TRIALS     = 50     # Optuna trials
N_CV_FOLDS   = 5      # Folds inside each Optuna trial
SEEDS        = [42, 2024, 777, 1337, 999]
N_SPLITS     = 10     # Folds per seed in ensemble
TOTAL_MODELS = len(SEEDS) * N_SPLITS
MAX_EPOCHS   = 200
PATIENCE     = 20

# Pre-computed fallback — update after first tuning run
PRECOMPUTED_PARAMS = {
    'n_blocks'    : 3,
    'd_block'     : 256,
    'dropout'     : 0.15,
    'lr'          : 1e-3,
    'weight_decay': 1e-5,
    'batch_size'  : 512,
}

## 1. Load & Preprocess Data

In [ ]:
print(f'Loading data from {DATA_DIR} ...')
train_df = pd.read_csv(os.path.join(DATA_DIR, 'train.csv'))
test_df  = pd.read_csv(os.path.join(DATA_DIR, 'test.csv'))
sub      = pd.read_csv(os.path.join(DATA_DIR, 'sample_submission.csv'))

print(f'Train : {train_df.shape}   Test : {test_df.shape}')
train_df.head(3)

In [ ]:
TARGET = 'Churn'

# Encode target if string
if train_df[TARGET].dtype == 'object':
    train_df[TARGET] = train_df[TARGET].map({'Yes': 1, 'No': 0, '1': 1, '0': 0})

# Combine train+test for joint label encoding (avoids unseen-category mismatch)
train_df['is_train'] = 1
test_df['is_train']  = 0
df = pd.concat([train_df, test_df], ignore_index=True)

features = [c for c in train_df.columns if c not in ['id', TARGET, 'is_train']]
categorical_features = []

print('Label-encoding categorical features ...')
for col in tqdm(features):
    if df[col].dtype == 'object':
        categorical_features.append(col)
        le = LabelEncoder()
        df[col] = le.fit_transform(df[col].fillna('Missing').astype(str))

num_features = [c for c in features if c not in categorical_features]
for col in num_features:
    df[col] = df[col].fillna(df[col].median())

train_enc = df[df['is_train'] == 1].reset_index(drop=True)
test_enc  = df[df['is_train'] == 0].reset_index(drop=True)

print(f'Categoricals : {categorical_features}')
print(f'Numerics     : {num_features}')

## 2. Feature Engineering

In [ ]:
def engineer_features(df):
    """Row-level stats + domain interactions on the three numeric columns."""
    df = df.copy()
    num_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']

    df['num_sum']  = df[num_cols].sum(axis=1)
    df['num_mean'] = df[num_cols].mean(axis=1)
    df['num_std']  = df[num_cols].std(axis=1)
    df['num_max']  = df[num_cols].max(axis=1)
    df['num_min']  = df[num_cols].min(axis=1)

    df['Average_Monthly_Actual'] = df['TotalCharges'] / (df['tenure'] + 1e-5)
    df['Monthly_diff']  = df['MonthlyCharges'] - df['Average_Monthly_Actual']
    df['Monthly_ratio'] = df['MonthlyCharges'] / (df['Average_Monthly_Actual'] + 1e-5)

    return df


X      = engineer_features(train_enc[features])
X_test = engineer_features(test_enc[features])
y      = train_enc[TARGET]

# One-hot encode low-cardinality categoricals (better than entity embeddings here)
print(f'One-hot encoding {len(categorical_features)} categorical features ...')
X      = pd.get_dummies(X,      columns=categorical_features, dtype=float)
X_test = pd.get_dummies(X_test, columns=categorical_features, dtype=float)

# Align columns (guard against rare category mismatch)
X, X_test = X.align(X_test, join='left', axis=1, fill_value=0)

D_IN = X.shape[1]
print(f'Train : {X.shape}  |  Test : {X_test.shape}  |  d_in = {D_IN}')
print(f'Target distribution : {y.value_counts().to_dict()}')

## 3. Training Utilities

In [ ]:
X_np      = X.values.astype(np.float32)
X_test_np = X_test.values.astype(np.float32)
y_np      = y.values.astype(np.float32)


def train_one_fold(
    X_tr, y_tr, X_val, y_val, X_tst,
    params,
    max_epochs=MAX_EPOCHS,
    patience=PATIENCE,
):
    """
    Train one RealMLP fold.
    Returns (val_probs, test_probs, best_val_auc).
    """
    scaler = StandardScaler()
    X_tr_s  = scaler.fit_transform(X_tr)
    X_val_s = scaler.transform(X_val)
    X_tst_s = scaler.transform(X_tst)

    def to_t(arr, label=False):
        t = torch.tensor(arr, dtype=torch.float32, device=DEVICE)
        return t.unsqueeze(1) if label else t

    X_tr_t  = to_t(X_tr_s)
    y_tr_t  = to_t(y_tr, label=True)
    X_val_t = to_t(X_val_s)
    X_tst_t = to_t(X_tst_s)

    train_dl = DataLoader(
        TensorDataset(X_tr_t, y_tr_t),
        batch_size=params['batch_size'],
        shuffle=True,
        drop_last=False,
    )

    model = MLP(
        d_in=X_tr_s.shape[1],
        d_out=1,
        n_blocks=params['n_blocks'],
        d_block=params['d_block'],
        dropout=params['dropout'],
    ).to(DEVICE)

    optimizer  = torch.optim.AdamW(model.parameters(), lr=params['lr'], weight_decay=params['weight_decay'])
    scheduler  = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max_epochs)
    criterion  = nn.BCEWithLogitsLoss()

    best_auc, best_state, wait = 0.0, None, 0

    for epoch in range(max_epochs):
        model.train()
        for xb, yb in train_dl:
            optimizer.zero_grad()
            criterion(model(xb), yb).backward()
            optimizer.step()
        scheduler.step()

        model.eval()
        with torch.no_grad():
            val_logits = model(X_val_t).squeeze(1).cpu().numpy()
        auc = roc_auc_score(y_val, 1 / (1 + np.exp(-val_logits)))

        if auc > best_auc:
            best_auc   = auc
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            wait = 0
        else:
            wait += 1
            if wait >= patience:
                break

    model.load_state_dict(best_state)
    model.eval()
    with torch.no_grad():
        val_probs  = torch.sigmoid(model(X_val_t).squeeze(1)).cpu().numpy()
        test_probs = torch.sigmoid(model(X_tst_t).squeeze(1)).cpu().numpy()

    del model, optimizer, scheduler, X_tr_t, y_tr_t, X_val_t, X_tst_t
    gc.collect()
    if DEVICE == 'cuda':
        torch.cuda.empty_cache()

    return val_probs, test_probs, best_auc


print('train_one_fold() ready.')

## 4. Optuna Hyperparameter Tuning

Set `RUN_TUNING = False` (top cell) to skip and use `PRECOMPUTED_PARAMS` instead.

In [ ]:
if RUN_TUNING:
    print(f'Optuna: {N_TRIALS} trials × {N_CV_FOLDS}-fold CV  |  device={DEVICE}')

    pbar = tqdm(total=N_TRIALS, desc='Optuna', unit='trial')
    best_so_far = [0.0]

    def objective(trial):
        params = {
            'n_blocks'    : trial.suggest_int('n_blocks', 1, 5),
            'd_block'     : trial.suggest_categorical('d_block', [64, 128, 256, 384, 512]),
            'dropout'     : trial.suggest_float('dropout', 0.0, 0.5),
            'lr'          : trial.suggest_float('lr', 1e-5, 1e-2, log=True),
            'weight_decay': trial.suggest_float('weight_decay', 1e-7, 1e-3, log=True),
            'batch_size'  : trial.suggest_categorical('batch_size', [256, 512, 1024, 2048]),
        }

        skf  = StratifiedKFold(n_splits=N_CV_FOLDS, shuffle=True, random_state=42)
        aucs = []

        for fold, (tr_idx, val_idx) in enumerate(skf.split(X_np, y_np)):
            _, _, auc = train_one_fold(
                X_np[tr_idx], y_np[tr_idx],
                X_np[val_idx], y_np[val_idx],
                X_test_np, params,
                max_epochs=100, patience=10,   # reduced for speed
            )
            aucs.append(auc)
            trial.report(np.mean(aucs), fold)
            if trial.should_prune():
                raise optuna.TrialPruned()

        score = np.mean(aucs)
        if score > best_so_far[0]:
            best_so_far[0] = score
        pbar.set_postfix({'AUC': f'{score:.5f}', 'Best': f'{best_so_far[0]:.5f}'})
        pbar.update(1)
        return score

    study = optuna.create_study(
        direction='maximize',
        pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=1),
    )
    study.optimize(objective, n_trials=N_TRIALS)
    pbar.close()

    best_params = study.best_trial.params
    print(f'\nBest CV AUC : {study.best_value:.5f}')
    for k, v in best_params.items():
        print(f'  {k:20s}: {v}')

    study.trials_dataframe().to_csv('realmlp_tuning_results.csv', index=False)
    print('Tuning results → realmlp_tuning_results.csv')

else:
    best_params = PRECOMPUTED_PARAMS
    print('Using PRECOMPUTED_PARAMS:')
    for k, v in best_params.items():
        print(f'  {k:20s}: {v}')

## 5. Multi-Seed Ensemble

5 seeds × 10 folds = **50 models** averaged for stable predictions.

In [ ]:
print(f'Ensemble: {len(SEEDS)} seeds × {N_SPLITS} folds = {TOTAL_MODELS} models\n')

test_preds_sum = np.zeros(len(X_test_np))
oof_sum        = np.zeros(len(X_np))
fold_auc_log   = []

for seed in tqdm(SEEDS, desc='Seeds'):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if DEVICE == 'cuda':
        torch.cuda.manual_seed_all(seed)

    skf      = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=seed)
    seed_oof = np.zeros(len(X_np))

    for fold, (tr_idx, val_idx) in enumerate(
        tqdm(skf.split(X_np, y_np), total=N_SPLITS, desc=f'  Seed {seed}', leave=False)
    ):
        val_preds, test_preds, fold_auc = train_one_fold(
            X_np[tr_idx], y_np[tr_idx],
            X_np[val_idx], y_np[val_idx],
            X_test_np, best_params,
        )
        fold_auc_log.append(fold_auc)
        seed_oof[val_idx] = val_preds
        oof_sum[val_idx] += val_preds
        test_preds_sum   += test_preds

    seed_auc = roc_auc_score(y_np, seed_oof)
    print(f'Seed {seed}  OOF AUC: {seed_auc:.5f}')

# ── Final metrics ─────────────────────────────────────────────────────────────
global_oof = oof_sum / len(SEEDS)
final_auc  = roc_auc_score(y_np, global_oof)

print(f'\n{"="*55}')
print(f'Fold AUC  mean={np.mean(fold_auc_log):.5f}  std={np.std(fold_auc_log):.5f}')
print(f'Global Ensemble OOF AUC : {final_auc:.5f}')
print(f'{"="*55}')

## 6. Save Submission & OOF

In [ ]:
# Submission
final_test_preds = test_preds_sum / TOTAL_MODELS
sub[TARGET]      = final_test_preds
sub_file         = 'submission_realmlp_standalone.csv'
sub.to_csv(sub_file, index=False)
print(f'Submission  → {sub_file}')
print(f'Pred range  : [{final_test_preds.min():.4f}, {final_test_preds.max():.4f}]')

# OOF (for blending in notebook 08/20)
oof_df = pd.DataFrame({'id': train_df['id'], 'oof_pred': global_oof})
oof_df.to_csv('oof_realmlp_standalone.csv', index=False)
print(f'OOF         → oof_realmlp_standalone.csv  (AUC={final_auc:.5f})')

sub.head()

## 7. Submit to Kaggle

In [ ]:
COMPETITION = 'playground-series-s6e3'
SUBMIT_FILE = sub_file   # 'submission_realmlp_standalone.csv'
SUBMIT_MSG  = f'RealMLP standalone {TOTAL_MODELS}-model ensemble, Optuna tuned, FE  |  OOF AUC={final_auc:.5f}'
AUTO_SUBMIT = False      # Set True to submit automatically

print(f'File    : {SUBMIT_FILE}')
print(f'Message : {SUBMIT_MSG}')

if AUTO_SUBMIT:
    result = subprocess.run(
        ['kaggle', 'competitions', 'submit',
         '-c', COMPETITION,
         '-f', SUBMIT_FILE,
         '-m', SUBMIT_MSG],
        capture_output=True, text=True,
    )
    print(result.stdout)
    if result.returncode != 0:
        print('STDERR:', result.stderr)
else:
    print('\nRun this to submit:')
    print(f'  kaggle competitions submit -c {COMPETITION} -f {SUBMIT_FILE} -m "{SUBMIT_MSG}"')

### To blend with XGBoost / LightGBM
Add to `notebooks/08_blend_submissions.ipynb`:
```python
{'file': 'submission_realmlp_standalone.csv', 'label': 'RealMLP', 'public_score': 0.0},
```